<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: Test your own answer

**[Test your own answer](https://learning.nextia-ai.com/courses/experiments/m06/test-your-own-answer/)** · Experiments and Causal Impact: Did Our Change Work? · Module 6, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** try to break your own estimate with three refutation tests (placebo treatment, random common cause, data subsets), see that they pass on a biased estimate, and ask how strong a hidden confounder would need to be to change the answer.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Regression adjustment, from [Regression adjustment](https://learning.nextia-ai.com/courses/experiments/m06/regression-adjustment/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m06/test-your-own-answer/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M06-L03-test-your-own-answer/test-your-own-answer-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scipy==1.15.3 statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 2 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "logs/customers.csv.gz": "0e9bd54ac28ef98bf94f6d1d0abe1bf0f8e51f67384d7b08c94b0b9310ab320a",
    "logs/truth.json": "8e2cd0eda8b31c1fba87687dfaf14b598d24921cf38734e8d18a21d8126b009f"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The estimate to test

The regression with the six confounders, as in the first lesson of this module.

In [ ]:
logs = pd.read_csv("data/logs/customers.csv.gz")
logs["desktop"] = (logs.device == "desktop").astype(int)
CONFOUNDERS = "pre_spend + pre_orders + visits + tenure_days + new_customer + desktop"

def estimate(data, treatment="used_block", extra=""):
    m = smf.ols(f"revenue ~ {treatment} + {CONFOUNDERS}{extra}", data=data).fit()
    return float(m.params[treatment])

est = estimate(logs)
print(f"estimate: {est:+.2f} dollars per customer")

> **Check.** You should see `estimate: +3.38 dollars per customer`.

## 2. Placebo treatment

Shuffle the column `used_block`, so that "users" are chosen at random. A random "treatment" has no effect, so a sound method should give about 0.

In [ ]:
rng = np.random.default_rng(1)
placebo = []
for _ in range(20):
    fake = logs.assign(fake_block=rng.permutation(logs.used_block.to_numpy()))
    placebo.append(estimate(fake, "fake_block"))
print(f"placebo: mean {np.mean(placebo):+.2f}, from {min(placebo):+.2f} to {max(placebo):+.2f}")

> **Check.** You should see `placebo: mean +0.03, from -1.25 to +1.04`.

## 3. Random common cause

Add a column of random numbers as if it were one more confounder. Noise causes nothing, so the estimate should not move.

In [ ]:
rcc = [estimate(logs.assign(noise=rng.normal(size=len(logs))), extra=" + noise") for _ in range(20)]
print(f"random common cause: mean {np.mean(rcc):+.2f}")

> **Check.** You should see `random common cause: mean +3.38`.

## 4. Data subsets

Estimate again on 20 random samples of 80% of the customers. A sound estimate should not depend on a few customers.

In [ ]:
subset = [estimate(logs.sample(frac=0.8, random_state=int(rng.integers(1_000_000)))) for _ in range(20)]
print(f"data subsets: mean {np.mean(subset):+.2f}, from {min(subset):+.2f} to {max(subset):+.2f}")
t = truth("logs")
print(f"truth: {t['ate']['effect']:+.2f}; the hidden confounder: {t['hidden_confounder_size']}")

> **Check.** You should see `data subsets: mean +3.31, from +2.11 to +3.90` and `truth: +2.37; the hidden confounder: {'share_gap': 0.0872, 'dollars': 10.27}`.

All three tests pass, and the estimate is still about 1 dollar too high. The tests check the method and the data you have. None of them can see a column that is not in the data, like the garden.

## 5. How big a hidden confounder? First, the rule on a toy

Make up a 0/1 column that 60% of users and 50% of non-users have (a share gap of 0.10), and that adds 10 dollars of revenue. The naive difference should move by about 0.10 × 10 = 1 dollar.

In [ ]:
toy = logs.copy()
p = np.where(toy.used_block == 1, 0.60, 0.50)
toy["made_up"] = (rng.random(len(toy)) < p).astype(int)
gap = toy.loc[toy.used_block == 1, "made_up"].mean() - toy.loc[toy.used_block == 0, "made_up"].mean()
naive = lambda d: d.loc[d.used_block == 1, "revenue"].mean() - d.loc[d.used_block == 0, "revenue"].mean()
toy_plus = toy.assign(revenue=toy.revenue + 10 * toy.made_up)
print(f"share gap {gap:.3f}; the naive difference moved by {naive(toy_plus) - naive(toy):+.2f} (rule: {gap * 10:+.2f})")

> **Check.** You should see `share gap 0.103; the naive difference moved by +1.03 (rule: +1.03)`.

The rule is exact for a plain difference. For a regression it holds about, for the part of the hidden column that the other columns do not explain. The grid below uses it.

In [ ]:
grid = pd.DataFrame({f"{d} dollars": [round(est - g * d, 2) for g in (0.05, 0.10, 0.20, 0.30)]
                     for d in (5, 10, 20, 40)}, index=[f"gap {g:.2f}" for g in (0.05, 0.10, 0.20, 0.30)])
grid

Each cell is the estimate that would remain if a hidden 0/1 confounder had that share gap and that effect in dollars. The simulator's garden has a share gap of 0.087 and is worth 10.27 dollars: near the cell (0.10, 10 dollars), which leaves about the truth.

## 6. Your turn: what would change the decision?

Omar keeps the block if its effect is above 0. Suppose a hidden confounder adds 10 dollars of revenue.

> **Your turn.** What share gap would bring the estimate `est` exactly to 0? Put it, rounded to 3 decimals, in `gap_needed`.

In [ ]:
gap_needed = round(est / 10, 3)
print(gap_needed)

In [ ]:
expect('gap_needed', gap_needed, 0.338)

A hidden confounder worth 10 dollars would need to be about 34 points more common among users than among non-users. The garden's gap is under 9 points. So the sign of the answer is safe; its size is not.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m06/test-your-own-answer/) has the charts and the knowledge checks that count toward your certificate. In the project, `xp refute` runs the same three tests with DoWhy, and prints the same grid.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [The average hides the people](https://learning.nextia-ai.com/courses/experiments/m07/the-average-hides-the-people/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m06/test-your-own-answer/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.